# [STARTER] Exercise - Output structured Agent responses

In this exercise, you'll learn how to enhance your AI agent to provide structured outputs using Pydantic models. This will help ensure the agent's responses are consistent, validated, and easily usable in downstream applications.

## Challenge

You have an existing Agent class that can:
- Process user messages
- Use tools when needed
- Generate responses

Now you need to enhance it to:
- Define structured output formats using Pydantic
- Parse and validate responses
- Return data in a consistent JSON format



## Setup
First, let's import the necessary libraries:

In [2]:
from typing import List, Dict, Any, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import json

from lib.messages import BaseMessage, UserMessage, SystemMessage, ToolMessage
from lib.tooling import tool
from lib.llm import LLM
from lib.parsers import PydanticOutputParser, JsonOutputParser

## Defining Structured Output Models

Let's create a Pydantic model for a meeting summary with action items:


In [ ]:
class ActionItem(BaseModel):
    """A single follow-up task assigned during a meeting"""
    task: Annotated[str, Field(description="Description of the task to be completed")]
    assignee: Annotated[str, Field(description="Name of the person responsible for the task")]
    due_date: Annotated[str, Field(description="When the task is due, e.g. 'next Friday' or '2024-03-22'")]

In [ ]:
class MeetingSummary(BaseModel):
    """A summary of a meeting, including key points, titles, dates, and action items"""
    title: Annotated[str, Field(description="The title of the meeting")]
    date: Annotated[str, Field(description="The date of the meeting, e.g. '2024-03-15'")]
    participants: Annotated[List[str], Field(description="List of participants in the meeting")]
    key_points: Annotated[List[str], Field(description="List of key points discussed during the meeting")]
    action_items: Annotated[List[ActionItem], Field(description="List of action items assigned during the meeting")]

## Enhanced Agent Class

Now let's create an enhanced version of our Agent class that supports structured outputs:


In [5]:
class StructuredAgent:
    """An AI Agent that provides structured outputs"""
    
    def __init__(
        self,
        role: str = "Meeting Assistant",
        instructions: str = "Help summarize meetings and track action items",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[Any] = None,
        output_model: BaseModel = None
    ):
        """Initialize the agent with its configuration
        
        Args:
            role: The agent's role/persona
            instructions: Basic instructions for the agent
            model: The LLM model to use
            temperature: Creativity parameter (0.0 = more deterministic)
            tools: List of tools the agent can use
            output_model: Pydantic model for structured output
        """
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools or []
        self.output_model = output_model
        load_dotenv()  # Load environment variables from .env file
        self.llm = LLM(
            model=self.model, 
            temperature=self.temperature, 
            tools=self.tools)
       


    def invoke(self, user_message: str) -> dict:
        """Process a user message and return a structured response
        
        Args:
            user_message: The user's input message
            
        Returns:
            A dictionary containing the structured response
        """
        # Build the system prompt; if an output model is set, include its JSON schema
        system_prompt = f"You are a {self.role}. {self.instructions}"
        if self.output_model:
            schema = json.dumps(self.output_model.model_json_schema(), indent=2)
            system_prompt += (
                "\n\nRespond ONLY with a JSON object that conforms to this JSON schema. "
                "Do not include any explanation or markdown code fences.\n"
                f"{schema}"
            )

        messages = [
            SystemMessage(content=system_prompt),
            UserMessage(content=user_message),
        ]

        ai_message = self.llm.invoke(messages)

        # Strip markdown fences in case the model wraps its JSON anyway
        content = ai_message.content.strip()
        if content.startswith("```"):
            ai_message.content = content.strip("`").removeprefix("json").strip()

        # Parse and validate against the Pydantic model, then return a plain dict
        if self.output_model:
            parser = PydanticOutputParser(model_class=self.output_model)
            return parser.parse(ai_message).model_dump()

        return {"content": ai_message.content}

## Testing the Structured Agent

Let's test our enhanced agent with a meeting summary example:


In [6]:
# Create an agent instance with the MeetingSummary model
meeting_agent = StructuredAgent(
    role="Meeting Assistant",
    instructions="Summarize meetings and track action items in a structured format",
    output_model=MeetingSummary
)

In [7]:
meeting_transcript = """
Project Planning Meeting - March 15, 2024

Attendees: John, Sarah, Mike

Discussion:
- Reviewed Q1 project timeline
- Discussed resource allocation
- Identified potential risks

Next steps:
1. John will update the project plan by next Friday
2. Sarah needs to coordinate with the design team by Wednesday
3. Mike will prepare the risk assessment document by end of month
"""

In [8]:
summary = meeting_agent.invoke(meeting_transcript)
print(json.dumps(summary, indent=2))

{
  "title": "Project Planning Meeting",
  "date": "2024-03-15",
  "participants": [
    "John",
    "Sarah",
    "Mike"
  ],
  "key_points": [
    "Reviewed Q1 project timeline",
    "Discussed resource allocation",
    "Identified potential risks"
  ],
  "action_items": [
    {
      "task": "Update the project plan",
      "assignee": "John",
      "due_date": "2024-03-22"
    },
    {
      "task": "Coordinate with the design team",
      "assignee": "Sarah",
      "due_date": "2024-03-20"
    },
    {
      "task": "Prepare the risk assessment document",
      "assignee": "Mike",
      "due_date": "2024-03-31"
    }
  ]
}


## Validating the Output

Let's verify that our output matches our Pydantic model structure:


In [9]:
# Create a MeetingSummary instance from the output
validated_summary = MeetingSummary(**summary)

In [10]:
# Access structured data
print("Meeting Title:", validated_summary.title)
print("\nParticipants:")
for participant in validated_summary.participants:
    print(f"- {participant}")

print("\nAction Items:")
for item in validated_summary.action_items:
    print(f"- {item.task} (Assigned to: {item.assignee}, Due: {item.due_date})")

Meeting Title: Project Planning Meeting

Participants:
- John
- Sarah
- Mike

Action Items:
- Update the project plan (Assigned to: John, Due: 2024-03-22)
- Coordinate with the design team (Assigned to: Sarah, Due: 2024-03-20)
- Prepare the risk assessment document (Assigned to: Mike, Due: 2024-03-31)
